In [1]:
import pandas as pd
import os
from pathlib import Path
import glob
import re

In [2]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession

In [3]:
DIRECTORY_RAW = '/data/raw/despesas_contabeis/data/raw/'
DIRECTORY_SILVER = '/data/raw/despesas_contabeis/data/silver/'

In [4]:
spark = SparkSession.builder.appName("Preprocessamento").getOrCreate()

In [5]:
def load_datas_spark(files):
    dffinal = spark.read.format("parquet").load(files[0])
    print(dffinal.printSchema())
    for f in files[1:]:
        print(f"Processando: {f}")
        try:
            results = spark.read.parquet(f)
            if "VL_SALDO_INICIAL" not in results.columns:
                results = results.withColumn("VL_SALDO_INICIAL", F.lit(0.0))
                results = results.select(*dffinal.columns)

            dffinal = dffinal.unionByName(results)
        except Exception as e:
            
            print(f"Erro ao processar {f}: {e}")
    return dffinal

In [6]:
files_path = glob.glob(DIRECTORY_RAW+'*.parquet')

filter_paths_years = [str(y) for y in range(2019, 2025)]

files_path_filter = [f for f in files_path if any(fp in f for fp in filter_paths_years)]

In [7]:
#files_path_filter
len(files_path)

files_path.sort(reverse=True)

In [8]:

print("Qtd de files:", len(files_path))
df = load_datas_spark(files_path)

Qtd de files: 42
root
 |-- DATA: string (nullable = true)
 |-- REG_ANS: long (nullable = true)
 |-- CD_CONTA_CONTABIL: long (nullable = true)
 |-- DESCRICAO: string (nullable = true)
 |-- VL_SALDO_INICIAL: string (nullable = true)
 |-- VL_SALDO_FINAL: string (nullable = true)

None
Processando: /data/raw/despesas_contabeis/data/raw/4T2023.parquet
Processando: /data/raw/despesas_contabeis/data/raw/4T2022.parquet
Processando: /data/raw/despesas_contabeis/data/raw/4T2021.parquet
Processando: /data/raw/despesas_contabeis/data/raw/4T2020.parquet
Processando: /data/raw/despesas_contabeis/data/raw/4T2019.parquet
Processando: /data/raw/despesas_contabeis/data/raw/4T2018.parquet
Processando: /data/raw/despesas_contabeis/data/raw/4T2017.parquet
Processando: /data/raw/despesas_contabeis/data/raw/4T2016.parquet
Processando: /data/raw/despesas_contabeis/data/raw/4T2015.parquet
Processando: /data/raw/despesas_contabeis/data/raw/3T2024.parquet
Processando: /data/raw/despesas_contabeis/data/raw/3T2023

In [9]:
df.count()

28873711

In [10]:
df.printSchema()

root
 |-- DATA: string (nullable = true)
 |-- REG_ANS: double (nullable = true)
 |-- CD_CONTA_CONTABIL: double (nullable = true)
 |-- DESCRICAO: string (nullable = true)
 |-- VL_SALDO_INICIAL: string (nullable = true)
 |-- VL_SALDO_FINAL: string (nullable = true)



In [11]:
df.show(5)

+----------+--------+-----------------+--------------------+----------------+--------------+
|      DATA| REG_ANS|CD_CONTA_CONTABIL|           DESCRICAO|VL_SALDO_INICIAL|VL_SALDO_FINAL|
+----------+--------+-----------------+--------------------+----------------+--------------+
|2024-10-01|420051.0|     4.71119017E8|ProvisÃ£o para De...|               0|             0|
|2024-10-01|420051.0|     4.71119018E8|     Outras Despesas|               0|             0|
|2024-10-01|420051.0|     4.71119019E8|(-) RecuperaÃ§Ã£o...|               0|             0|
|2024-10-01|420051.0|           4712.0|AJUSTES NEGATIVOS...|               0|             0|
|2024-10-01|420051.0|          47121.0|AJUSTES NEGATIVOS...|               0|             0|
+----------+--------+-----------------+--------------------+----------------+--------------+
only showing top 5 rows



In [12]:
df = (
    df
    .withColumn("REG_ANS", F.col("REG_ANS").cast("int"))


    .withColumn("CD_CONTA_CONTABIL", F.col("CD_CONTA_CONTABIL").cast('int').cast("string"))

    .withColumn("DATA", F.to_date(F.col("DATA"), "yyyy-MM-dd"))


    .withColumn("VL_SALDO_INICIAL", F.regexp_replace("VL_SALDO_INICIAL", r"\.", ""))   
    .withColumn("VL_SALDO_INICIAL", F.regexp_replace("VL_SALDO_INICIAL", ",", "."))    
    .withColumn("VL_SALDO_INICIAL", F.col("VL_SALDO_INICIAL").cast("double"))

    .withColumn("VL_SALDO_FINAL", F.regexp_replace("VL_SALDO_FINAL", r"\.", ""))
    .withColumn("VL_SALDO_FINAL", F.regexp_replace("VL_SALDO_FINAL", ",", "."))
    .withColumn("VL_SALDO_FINAL", F.col("VL_SALDO_FINAL").cast("double"))
)

In [13]:
def corrigir_encoding(texto):
    """
    Corrige problemas comuns de encoding em textos portugueses
    """
    if any(re.findall('[\x80-\x90]', texto)) or ('Ã§' in texto or 'Ã£' in texto or 'Ã¡' in texto or 'Ã' in texto or 'Ã' in texto):
        try:
            print("caracter especial encontrado")
            # Corrige dupla codificação (UTF-8 → Latin-1 → UTF-8)
            return texto.encode('latin-1').decode('utf-8').strip()
        except (UnicodeEncodeError, UnicodeDecodeError) as e:
            print(f"Erro ao corrigir '{texto}': {e}")
            try:
            # Se falhar, tenta outras abordagens
                return texto.encode('utf-8').decode('latin-1').strip()
            except:
                return texto.strip()
    print("retorno sem tratamento")
    return texto.strip()

In [14]:
udf_corrigir_encoding = F.udf(corrigir_encoding, T.StringType())
df = df.withColumn('DESCRICAO', udf_corrigir_encoding(F.col('DESCRICAO')))

In [15]:

df.select(F.count(F.when(F.col('REG_ANS').isNull(), 'REG_ANS')).alias('REG_ANS')).show(5)
df = df.where(F.col('REG_ANS').isNotNull())

+-------+
|REG_ANS|
+-------+
| 155525|
+-------+



In [16]:
df.rdd.getNumPartitions()

140

In [ ]:
df.limit(10000).write.mode('overwrite').option("maxRecordsPerFile", 500_000).parquet(DIRECTORY_SILVER+'tb_union_dados_contabeis')

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 35564)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

In [ ]:
df.select('VL_SALDO_FINAL').describe().show()

+-------+--------------------+
|summary|      VL_SALDO_FINAL|
+-------+--------------------+
|  count|            28718186|
|   mean|   7604803.435033823|
| stddev|1.6601850795341676E8|
|    min|  -1.424000909082E10|
|    max|   3.827532591681E10|
+-------+--------------------+



In [ ]:
display(df.show(5))

+----------+-------+-----------------+--------------------+----------------+--------------+
|      DATA|REG_ANS|CD_CONTA_CONTABIL|           DESCRICAO|VL_SALDO_INICIAL|VL_SALDO_FINAL|
+----------+-------+-----------------+--------------------+----------------+--------------+
|2024-10-01| 420051|        471119017|ProvisÃ£o para De...|             0.0|           0.0|
|2024-10-01| 420051|        471119018|     Outras Despesas|             0.0|           0.0|
|2024-10-01| 420051|        471119019|(-) RecuperaÃ§Ã£o...|             0.0|           0.0|
|2024-10-01| 420051|             4712|AJUSTES NEGATIVOS...|             0.0|           0.0|
|2024-10-01| 420051|            47121|AJUSTES NEGATIVOS...|             0.0|           0.0|
+----------+-------+-----------------+--------------------+----------------+--------------+
only showing top 5 rows



None

In [15]:
files_path_operadoras = glob.glob('/data/raw/operadoras/*')


In [16]:
dfoperadoras = spark.read.csv('/data/raw/operadoras/', header=True, sep=',', encoding='utf-8', inferSchema=True)
print(dfoperadoras.count())
print(dfoperadoras.printSchema())

4138
root
 |-- _c0: integer (nullable = true)
 |-- Registro_ANS: string (nullable = true)
 |-- CNPJ: string (nullable = true)
 |-- Razao_Social: string (nullable = true)
 |-- Nome_Fantasia: string (nullable = true)
 |-- Modalidade: string (nullable = true)
 |-- Logradouro: string (nullable = true)
 |-- Numero: string (nullable = true)
 |-- Complemento: string (nullable = true)
 |-- Bairro: string (nullable = true)
 |-- Cidade: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- CEP: string (nullable = true)
 |-- DDD: string (nullable = true)
 |-- Telefone: string (nullable = true)
 |-- Fax: string (nullable = true)
 |-- Endereco_eletronico: string (nullable = true)
 |-- Representante: string (nullable = true)
 |-- Cargo_Representante: string (nullable = true)
 |-- Regiao_de_Comercializacao: string (nullable = true)
 |-- Data_Registro_ANS: string (nullable = true)
 |-- Data_Descredenciamento: string (nullable = true)
 |-- Motivo_do_Descredenciamento: string (nullable = true)

In [17]:
dfoperadoras.show(5)

+---+------------+--------------+--------------------+--------------------+--------------------+--------------------+------+-----------+------------+------------------+---+--------+---+--------+--------+--------------------+--------------------+--------------------+-------------------------+-----------------+----------------------+---------------------------+
|_c0|Registro_ANS|          CNPJ|        Razao_Social|       Nome_Fantasia|          Modalidade|          Logradouro|Numero|Complemento|      Bairro|            Cidade| UF|     CEP|DDD|Telefone|     Fax| Endereco_eletronico|       Representante| Cargo_Representante|Regiao_de_Comercializacao|Data_Registro_ANS|Data_Descredenciamento|Motivo_do_Descredenciamento|
+---+------------+--------------+--------------------+--------------------+--------------------+--------------------+------+-----------+------------+------------------+---+--------+---+--------+--------+--------------------+--------------------+--------------------+----------

In [18]:
print(dfoperadoras.columns)

['_c0', 'Registro_ANS', 'CNPJ', 'Razao_Social', 'Nome_Fantasia', 'Modalidade', 'Logradouro', 'Numero', 'Complemento', 'Bairro', 'Cidade', 'UF', 'CEP', 'DDD', 'Telefone', 'Fax', 'Endereco_eletronico', 'Representante', 'Cargo_Representante', 'Regiao_de_Comercializacao', 'Data_Registro_ANS', 'Data_Descredenciamento', 'Motivo_do_Descredenciamento']


In [19]:
dfoperadoras = dfoperadoras.select('Registro_ANS', 'Razao_Social', 'Nome_Fantasia', 'Modalidade', 'UF',  'Data_Registro_ANS', 'Data_Descredenciamento', 'Motivo_do_Descredenciamento')

+-------+
|REG_ANS|
+-------+
| 155525|
+-------+



In [21]:
dfjoin = df.join(dfoperadoras, df.REG_ANS == dfoperadoras.Registro_ANS, 'left')
dfjoin.groupBy('Modalidade').count().orderBy(F.desc('count')).show(10)

+--------------------+--------+
|          Modalidade|   count|
+--------------------+--------+
|  Cooperativa Médica|10254027|
|   Medicina de Grupo| 7633472|
|Administradora de...| 3413365|
|          Autogestão| 2733375|
|Odontologia de Grupo| 2284897|
|Cooperativa odont...| 1141821|
|         Filantropia|  988434|
|Seguradora Especi...|  268795|
+--------------------+--------+



In [22]:
dfjoin.show(10)

+----------+-------+-----------------+--------------------+----------------+--------------+------------+--------------------+--------------------+--------------------+---+-----------------+----------------------+---------------------------+
|      DATA|REG_ANS|CD_CONTA_CONTABIL|           DESCRICAO|VL_SALDO_INICIAL|VL_SALDO_FINAL|Registro_ANS|        Razao_Social|       Nome_Fantasia|          Modalidade| UF|Data_Registro_ANS|Data_Descredenciamento|Motivo_do_Descredenciamento|
+----------+-------+-----------------+--------------------+----------------+--------------+------------+--------------------+--------------------+--------------------+---+-----------------+----------------------+---------------------------+
|2024-10-01| 420051|        471119017|ProvisÃ£o para De...|             0.0|           0.0|      420051|SMART CARE SISTEM...|SMARTCARE ODONTOL...|Odontologia de Grupo| SP|       2016-02-03|                  NULL|                       NULL|
|2024-10-01| 420051|        47111901

In [23]:
dfjoin = dfjoin.withColumn('size_cd_contabil', F.length(F.col('CD_CONTA_CONTABIL')))

In [24]:
dfjoin.groupBy('size_cd_contabil').count().orderBy(F.desc('count')).show(10)

+----------------+--------+
|size_cd_contabil|   count|
+----------------+--------+
|               9|11162643|
|               8| 5437601|
|               6| 3462536|
|               5| 3305898|
|               4| 2943027|
|               3| 1604856|
|               2|  607844|
|               1|  193781|
+----------------+--------+



In [25]:
import unidecode
import re

In [26]:
def corrigir_encoding(texto):
    """
    Corrige problemas comuns de encoding em textos portugueses
    """
    if any(re.findall('[\x80-\x90]', texto)) or ('Ã§' in texto or 'Ã£' in texto or 'Ã¡' in texto or 'Ã' in texto or 'Ã' in texto):
        try:
            print("caracter especial encontrado")
            # Corrige dupla codificação (UTF-8 → Latin-1 → UTF-8)
            return texto.encode('latin-1').decode('utf-8').strip()
        except (UnicodeEncodeError, UnicodeDecodeError) as e:
            print(f"Erro ao corrigir '{texto}': {e}")
            try:
            # Se falhar, tenta outras abordagens
                return texto.encode('utf-8').decode('latin-1').strip()
            except:
                return texto.strip()
    print("retorno sem tratamento")
    return texto.strip()

In [28]:
dfjoin.groupBy('Registro_ANS', 'Razao_Social').agg(F.countDistinct('size_cd_contabil').alias('qtd_sizes')).groupBy('qtd_sizes').count().orderBy(F.desc('count')).show(10)

+---------+-----+
|qtd_sizes|count|
+---------+-----+
|        8| 1544|
+---------+-----+



In [ ]:
dfplano_contas_a = dfjoin.select(F.col('CD_CONTA_CONTABIL').alias('n9_cd_contabil'), F.col('DESCRICAO_CORRIGIDA').alias('n9_ds_conta_contabil'), 'size_cd_contabil').distinct()

dfplano_contas_a.write.format('parquet').mode('overwrite').save('/data/raw/despesas_contabeis/data/raw/tb_plano_contas_bruto')

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 44408)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

Py4JError: An error occurred while calling o748.save

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [ ]:

dfplano_contas_b = (dfplano_contas_a.where(F.col('size_cd_contabil') == 9)
                    .withColumn('n1_cd_contabil', F.col('n9_ds_conta_contabil').substr(1, 1))
                    .withColumn('n2_cd_contabil', F.col('n9_ds_conta_contabil').substr(1, 2))
                    .withColumn('n3_cd_contabil', F.col('n9_ds_conta_contabil').substr(1, 3))
                    .withColumn('n4_cd_contabil', F.col('n9_ds_conta_contabil').substr(1, 4))
                    .withColumn('n5_cd_contabil', F.col('n9_ds_conta_contabil').substr(1, 5))
                    .withColumn('n6_cd_contabil', F.col('n9_ds_conta_contabil').substr(1, 6))
                    .withColumn('n8_cd_contabil', F.col('n9_ds_conta_contabil').substr(1, 8))
                    )

dfplano_contas_b.join(dfplano_contas_a, dfplano_contas_b.n9_cd_contabil == dfplano_contas_a.n9_cd_contabil, 'left').show(5)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 43096)
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3526, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_32135/1984386924.py", line 1, in <module>
    dfplano_contas_a.write.format('parquet').mode('overwrite').save('/data/raw/despesas_contabeis/data/raw/tb_plano_contas_bruto')
  File "/usr/local/spark/python/pyspark/sql/readwriter.py", line 1463, in save
    self._jwrite.save(path)
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1322, in __call__
    return_value = get_return_value(
                   ^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/pyspark/errors/exceptions/captured.py", line 179, in deco
    return f(*a, **kw)
           ^^^^^^^^^^^
  File "/usr/local/spark/pyth

ConnectionRefusedError: [Errno 111] Connection refused

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [30]:
dfjoin = (dfjoin
          .withColumn('n1_cd_contabil', F.col('CD_CONTA_CONTABIL').substr(1, 1))
          .withColumn('n2_cd_contabil', F.col('CD_CONTA_CONTABIL').substr(1, 2))
          .withColumn('n3_cd_contabil', F.col('CD_CONTA_CONTABIL').substr(1, 3))
          .withColumn('n4_cd_contabil', F.col('CD_CONTA_CONTABIL').substr(1, 4))
          .withColumn('n5_cd_contabil', F.col('CD_CONTA_CONTABIL').substr(1, 5))
          .withColumn('n6_cd_contabil', F.col('CD_CONTA_CONTABIL').substr(1, 6))
          .withColumn('n8_cd_contabil', F.col('CD_CONTA_CONTABIL').substr(1, 8))
          .withColumn('n9_cd_contabil', F.col('CD_CONTA_CONTABIL').substr(1, 9))
          )

In [33]:
dfjoin = dfjoin.filter('size_cd_contabil == 9')

In [34]:
dfjoin.show(10)

+----------+-------+-----------------+--------------------+----------------+--------------+------------+--------------------+--------------------+--------------------+---+-----------------+----------------------+---------------------------+----------------+--------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+
|      DATA|REG_ANS|CD_CONTA_CONTABIL|           DESCRICAO|VL_SALDO_INICIAL|VL_SALDO_FINAL|Registro_ANS|        Razao_Social|       Nome_Fantasia|          Modalidade| UF|Data_Registro_ANS|Data_Descredenciamento|Motivo_do_Descredenciamento|size_cd_contabil| DESCRICAO_CORRIGIDA|n1_cd_contabil|n2_cd_contabil|n3_cd_contabil|n4_cd_contabil|n5_cd_contabil|n6_cd_contabil|n8_cd_contabil|n9_cd_contabil|
+----------+-------+-----------------+--------------------+----------------+--------------+------------+--------------------+--------------------+--------------------+---+-----------------+-------------

In [35]:
dfjoin.count()

11162643

In [40]:
dfjoin = (dfjoin
          .join(dfplano_contas, dfjoin.n1_cd_contabil == dfplano_contas.CD_CONTA_CONTABIL_, how='left').withColumnRenamed('ds_conta_contabil', 'n1_ds_conta_contabil').drop('CD_CONTA_CONTABIL_')
          .join(dfplano_contas, dfjoin.n2_cd_contabil == dfplano_contas.CD_CONTA_CONTABIL_, how='left').withColumnRenamed('ds_conta_contabil', 'n2_ds_conta_contabil').drop('CD_CONTA_CONTABIL_')
          .join(dfplano_contas, dfjoin.n3_cd_contabil == dfplano_contas.CD_CONTA_CONTABIL_, how='left').withColumnRenamed('ds_conta_contabil', 'n3_ds_conta_contabil').drop('CD_CONTA_CONTABIL_')
          .join(dfplano_contas, dfjoin.n4_cd_contabil == dfplano_contas.CD_CONTA_CONTABIL_, how='left').withColumnRenamed('ds_conta_contabil', 'n4_ds_conta_contabil').drop('CD_CONTA_CONTABIL_')
          .join(dfplano_contas, dfjoin.n5_cd_contabil == dfplano_contas.CD_CONTA_CONTABIL_, how='left').withColumnRenamed('ds_conta_contabil', 'n5_ds_conta_contabil').drop('CD_CONTA_CONTABIL_')
          .join(dfplano_contas, dfjoin.n6_cd_contabil == dfplano_contas.CD_CONTA_CONTABIL_, how='left').withColumnRenamed('ds_conta_contabil', 'n6_ds_conta_contabil').drop('CD_CONTA_CONTABIL_')
          .join(dfplano_contas, dfjoin.n8_cd_contabil == dfplano_contas.CD_CONTA_CONTABIL_, how='left').withColumnRenamed('ds_conta_contabil', 'n8_ds_conta_contabil').drop('CD_CONTA_CONTABIL_')
          .join(dfplano_contas, dfjoin.n9_cd_contabil == dfplano_contas.CD_CONTA_CONTABIL_, how='left').withColumnRenamed('ds_conta_contabil', 'n9_ds_conta_contabil').drop('CD_CONTA_CONTABIL_')
          )

In [ ]:
dfjoin.count()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 51520)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

In [41]:
dfjoin.show(5)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 